# K-Means Clustering
## Real-world scenario: Segmenting mall customers

A shopping mall wants to group customers by **annual income** and **spending score** so marketing can target each group differently. There are **no labels** here - we want to *discover* groups - so this is **unsupervised learning**, and K-Means is the go-to method.

### Step 1 - Import the libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

np.random.seed(42)

### Step 2 - Create a small, realistic dataset
We simulate three natural customer groups (budget, average, premium shoppers). In real life we would NOT know these groups in advance - that is what K-Means will find.

In [ ]:
def group(income_mu, spend_mu, k):
    return np.column_stack([np.random.normal(income_mu, 8, k),
                            np.random.normal(spend_mu, 8, k)])

budget  = group(30, 25, 20)   # low income,  low spending
average = group(60, 55, 20)   # mid income,  mid spending
premium = group(90, 80, 20)   # high income, high spending

data = np.vstack([budget, average, premium])
df = pd.DataFrame(data, columns=['annual_income_k', 'spending_score']).round(1)

# Messy data on purpose
df.loc[3, 'spending_score'] = np.nan
df = pd.concat([df, df.iloc[[0]]], ignore_index=True)
df.head()

### Step 3 - Explore the data

In [ ]:
print('Shape:', df.shape)
print('\nMissing:\n', df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())
df.describe()

### Step 4 - Clean the data

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
df['spending_score'] = df['spending_score'].fillna(df['spending_score'].median())
print('Missing after cleaning:', df.isnull().sum().sum())

### Step 5 - Scale the features
K-Means uses distances, so we standardise income and spending to a common scale.

In [ ]:
X = df[['annual_income_k', 'spending_score']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### Step 6 - Choose the number of clusters (Elbow method)
We try k = 1..8 and plot the inertia. The 'elbow' where the curve bends suggests a good k.

In [ ]:
inertias = []
for k in range(1, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.plot(range(1, 9), inertias, 'o-')
plt.xlabel('Number of clusters (k)'); plt.ylabel('Inertia')
plt.title('Elbow method'); plt.show()

### Step 7 - Train K-Means with k = 3
The elbow suggests 3 clusters, matching our three customer types.

In [ ]:
model = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = model.fit_predict(X_scaled)
df.head()

### Step 8 - Evaluate the clustering
There are no labels to check against, so we use the **silhouette score** (ranges -1 to 1; higher = better separated clusters).

In [ ]:
score = silhouette_score(X_scaled, df['cluster'])
print('Silhouette score:', round(score, 3))
print('\nCustomers per cluster:\n', df['cluster'].value_counts())

### Step 9 - Visualise the segments

In [ ]:
plt.scatter(df['annual_income_k'], df['spending_score'],
            c=df['cluster'], cmap='viridis', s=50)
# plot the cluster centres (converted back to the original scale)
centres = scaler.inverse_transform(model.cluster_centers_)
plt.scatter(centres[:, 0], centres[:, 1], c='red', marker='X', s=200, label='Centres')
plt.xlabel('Annual income (k$)'); plt.ylabel('Spending score')
plt.title('Customer segments'); plt.legend(); plt.show()

### Step 10 - Assign a new customer to a segment
Income 55k, spending score 60:

In [ ]:
new_customer = pd.DataFrame({'annual_income_k': [55], 'spending_score': [60]})
new_scaled = scaler.transform(new_customer)
print('Assigned to cluster:', model.predict(new_scaled)[0])